In [1]:
import os
import re
import pandas as pd
import json

In [ ]:
#data_dir = "/home/nathan/aiesec/mc-activity/data/data.json"
os.chdir("..") #Execute just once per session
data_dir = os.path.join("data", "data.json")

In [3]:
with open(data_dir, encoding="utf-8") as f:
    raw = json.load(f)

# Flatten JSON response
response = raw["response"]

In [12]:
response.items()

dict_items([('o_matched_5', {'doc_count': 0, 'applicants': {'value': 0}}), ('i_realized_2', {'doc_count': 0, 'applicants': {'value': 0}}), ('i_realized_1', {'doc_count': 0, 'applicants': {'value': 0}}), ('o_matched_9', {'doc_count': 0, 'applicants': {'value': 0}}), ('o_matched_8', {'doc_count': 8, 'applicants': {'value': 6}}), ('o_matched_7', {'doc_count': 77, 'applicants': {'value': 66}}), ('i_realized_8', {'doc_count': 2, 'applicants': {'value': 2}}), ('i_realized_7', {'doc_count': 157, 'applicants': {'value': 157}}), ('i_realized_9', {'doc_count': 0, 'applicants': {'value': 0}}), ('o_an_accepted_9', {'doc_count': 0, 'applicants': {'value': 0}}), ('o_matched_2', {'doc_count': 0, 'applicants': {'value': 0}}), ('o_an_accepted_8', {'doc_count': 7, 'applicants': {'value': 5}}), ('o_matched_1', {'doc_count': 0, 'applicants': {'value': 0}}), ('o_an_accepted_7', {'doc_count': 34, 'applicants': {'value': 29}}), ('i_realized_5', {'doc_count': 0, 'applicants': {'value': 0}}), ('o_approval_brok

In [24]:
metric_rows = []

#Check metric patterns eg. o_matched_5 as seen from extracted JSON
#I also reffered form this doc:
#https://docs.google.com/presentation/d/1yGKTPk-lsmYDOfdO_qxzxXF0ZxLH_H-YCb-d-3tCxKU/edit?slide=id.g26ed0637951_0_79#slide=id.g26ed0637951_0_79

metric_pattern = re.compile(
    r"^(?P<direction>[io])_(?P<funnel_stage>.+)_(?P<product_id>\d+)$" #direction: i-incoming, o-outgoing
)

#Create table from top-level metrics
for metric_name, values in response.items():
    if not isinstance(values, dict):
        continue

    match = metric_pattern.match(metric_name)

    if match:
        metric_rows.append({
            "group": "overall",
            "metric": metric_name,
            "direction": match.group("direction"),
            "funnel_stage": match.group("funnel_stage"),
            "product_id": match.group("product_id"),
            "records": values.get("doc_count", 0),
            "applicants": values.get("applicants", {}).get("value", 0)
        })

Values from `product_id`:
- 1 => GV (old): Global volunteer
- 2 => GT (old): Global talent
- 5 => GE (old): Global enterpreneur
- 7 => GV (new): Gloabl volunteer
- 8 => GTa: Global teacher
- 9 => GT: Globale talent (professional)

In [25]:
metrics = pd.DataFrame(metric_rows)
metrics.head(10)

,group,metric,direction,funnel_stage,product_id,records,applicants
0,overall,o_matched_5,o,matched,5,0,0
1,overall,i_realized_2,i,realized,2,0,0
2,overall,i_realized_1,i,realized,1,0,0
3,overall,o_matched_9,o,matched,9,0,0
4,overall,o_matched_8,o,matched,8,8,6
5,overall,o_matched_7,o,matched,7,77,66
6,overall,i_realized_8,i,realized,8,2,2
7,overall,i_realized_7,i,realized,7,157,157
8,overall,i_realized_9,i,realized,9,0,0
9,overall,o_an_accepted_9,o,an_accepted,9,0,0
